# Lesson 1 - Preparing tabular data for classification

> This material is intended exclusively for educational purposes. The models and results must not be used for clinical diagnosis.

## Dataset

The dataset used as the foundation for this lesson is the [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic), which is widely used for breast cancer analysis and classification.

It contains computational measurements of features extracted from images of cell masses in mammography examinations. The features were computed from digitized images of fine needle aspirates (FNA) and are used to predict whether a mass is malignant (cancerous) or benign (noncancerous).

The data is available from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/) and was originally collected by Dr. William H. Wolberg at the University of Wisconsin Hospitals in Madison.

## Libraries and reproducibility

In [ ]:
import os
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Data acquisition

The compressed [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) data is downloaded directly from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/) and extracted into the `/data` folder.

In [ ]:
# Dataset URL
DATASET_URL = "https://archive.ics.uci.edu/static/public/17/breast+cancer+wisconsin+diagnostic.zip"

def find_project_root(start=Path.cwd()):
    """Walk through parent directories until the project root (containing pyproject.toml) is found."""
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = find_project_root()

# Destination folder where the archive will be extracted
dest_folder = PROJECT_ROOT / "data"

# Destination ZIP file
dest_zip_file = dest_folder / "breast_cancer_wisconsin_diagnostic.zip"

if not dest_folder.exists():

    # Create the folder if it does not exist
    print(f"Creating folder {dest_folder}...")
    dest_folder.mkdir(parents=True, exist_ok=True)

    print(f"Folder {dest_folder} created successfully.")

if not dest_zip_file.exists():

    # Download and save the file
    print("Downloading the dataset...")
    response = requests.get(DATASET_URL, timeout=60)  # Set a 60-second timeout
    response.raise_for_status()  # Raise an error if the download fails

    with open(dest_zip_file, "wb") as f:
        f.write(response.content)
    print(f"Download complete and saved to {dest_zip_file}")

    # Extract the archive
    print("Extracting the archive...")
    with zipfile.ZipFile(dest_zip_file, "r") as zip_ref:
        zip_ref.extractall(dest_folder)

    print(f"Files extracted to {dest_folder}")
else:
    print(f"The files already exist in {dest_folder}")

## Loading data and adjusting columns

The files are loaded, and a set of 12 columns is selected and renamed with more descriptive names to make the data easier to interpret.

In [ ]:
# Path to the data file
data_file = os.path.join(dest_folder, "wdbc.data")

# Load the data
data = pd.read_csv(data_file, header=None, sep=",")

# Select the first 12 columns
data = data.iloc[:, :12]

# Define column names
column_names = [
    "id_number",
    "diagnosis",
    "radius",
    "texture",
    "perimeter",
    "area",
    "smoothness",
    "compactness",
    "concavity",
    "concave points",
    "symmetry",
    "fractal dimension",
]

# Assign column names to the DataFrame
data.columns = column_names

# Display the first rows
data.head()

## Diagnosis distribution

Distribution of malignant and benign breast cancer cases based on the `diagnosis` column.

In [ ]:
# Display counts for the "diagnosis" column
data["diagnosis"].value_counts()

Distribution of attributes by diagnosis, using `radius`, `texture`, `perimeter`, and `area`, grouped by malignant and benign diagnoses.

In [ ]:
# Plot attribute distributions by diagnosis
sns.pairplot(
    data,
    hue="diagnosis",
    vars=["radius", "texture", "perimeter", "area"],
    palette="Set2",
)
plt.suptitle("Attribute Distribution by Diagnosis", y=1.02)
plt.show()

## Save the data

The DataFrame is saved as a Parquet file in the `data` folder.

In [ ]:
# Destination Parquet file
dest_parquet_file = dest_folder / "breast_cancer.parquet"

# Save the DataFrame as a Parquet file in the data folder
data.to_parquet(dest_parquet_file)

## Generate synthetic reports from tabular data

Synthetic medical reports are generated in English by combining fictional patient data from `Faker` with the features and diagnosis of each examination.

In [ ]:
from faker import Faker

fake = Faker("en_US")


def translate_diagnosis(diagnosis):
    """Translate the diagnosis code ('B'/'M') into an English label."""
    return "Benign" if diagnosis == "B" else "Malignant"


def generate_report(row):
    """Generate a synthetic medical report in English from one DataFrame row."""
    patient = fake.name()
    size = round(row["radius"] * 2, 1)
    margin_texture = "regular" if row["texture"] < 20 else "irregular"
    quadrant = random.choice([
        "upper left", "upper right", "lower left", "lower right"
    ])
    diagnosis_label = translate_diagnosis(row["diagnosis"])

    return f"""
        Patient: {patient}
        Examination: Mammography
        Examination Date: {fake.date_this_year()}

        Description:
        A lesion measuring approximately {size} mm was observed in the {quadrant} quadrant, with {margin_texture} margins.
        The examination suggests that the lesion has {diagnosis_label.lower()} characteristics.

        Conclusion: {diagnosis_label}.
        Recommendation: {("Follow up with another examination in 6 months" if diagnosis_label == "Benign" else "Refer for biopsy and oncological evaluation")}.
        """


# Apply the function to every row in the dataset
data["report"] = data.apply(generate_report, axis=1)

## Inspect the generated reports

Inspect the generated synthetic reports by displaying concise examples and the complete text for malignant and benign cases.

In [ ]:
# Display only the diagnosis and report columns
data[["id_number", "diagnosis", "report"]].head()

Display the complete reports for the first three malignant cases.

In [ ]:
# Display the complete reports for the first three malignant diagnoses
for report in data[data["diagnosis"] == "M"]["report"].head(3):
    print(report)
    print("=" * 80)

Display the complete reports for the first three benign cases.

In [ ]:
# Display the complete reports for the first three benign diagnoses
for report in data[data["diagnosis"] == "B"]["report"].head(3):
    print(report)
    print("=" * 80)

## Save the data

The DataFrame containing the reports is saved as a Parquet file in the `data` folder.

In [ ]:
# Destination Parquet file
dest_parquet_file = dest_folder / "breast_cancer_report.parquet"

# Save the DataFrame as a Parquet file in the data folder
data.to_parquet(dest_parquet_file)

## Add noise

Optionally add noise to the data by randomly changing the diagnosis of a small fraction of samples to simulate real-world imperfections.

In [ ]:
NOISE = True  # Set to True to add noise to the data


def add_noise(row):
    """Add noise to one DataFrame row, with a chance of flipping its diagnosis."""

    # row["radius"] += random.uniform(-1, 1)  # Small random radius adjustment
    # row["texture"] += random.uniform(-1, 1)  # Small random texture adjustment

    if random.random() < 0.05:  # 5% chance of changing the diagnosis
        row["diagnosis"] = "B" if row["diagnosis"] == "M" else "M"

    return row


if NOISE:
    data = data.apply(add_noise, axis=1)

## Save the data

The DataFrame containing reports and noise is saved as a Parquet file in the `data` folder.

In [ ]:
if NOISE:
    # Destination Parquet file
    dest_parquet_file = dest_folder / "breast_cancer_noise.parquet"

    # Save the DataFrame as a Parquet file in the data folder
    data.to_parquet(dest_parquet_file)